# Credit Scoring Model — Complete Data Analysis & Machine Learning

**Dataset:** 100,000 training records | 50,000 test records  
**Goal:** Predict creditworthiness as **Good**, **Standard**, or **Poor**  
**Models:** Logistic Regression · Decision Tree · Random Forest  

---

## 0. Setup & Imports

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib, json
from pathlib import Path

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, confusion_matrix, classification_report, roc_curve
)
from sklearn.preprocessing import label_binarize

pd.set_option('display.max_columns', None)
sns.set_theme(style='whitegrid', font_scale=1.1)

PALETTE = {'Good': '#2ecc71', 'Standard': '#f39c12', 'Poor': '#e74c3c'}
FIG_DIR = Path('../outputs/figures')
FIG_DIR.mkdir(parents=True, exist_ok=True)

print('Libraries loaded successfully!')

## 1. Load & Inspect Datasets

In [ ]:
train = pd.read_csv('../data/train.csv', low_memory=False)
test  = pd.read_csv('../data/test.csv',  low_memory=False)

print(f'Train shape : {train.shape}')
print(f'Test shape  : {test.shape}')
train.head(3)

In [ ]:
print('Column data types:')
print(train.dtypes)

In [ ]:
print('Basic statistics:')
train.describe(include='all').T

## 2. Missing Values & Duplicates

In [ ]:
miss = train.isnull().sum()
miss_pct = (miss / len(train) * 100).round(2)
missing_df = pd.DataFrame({'Missing': miss, 'Pct%': miss_pct})
missing_df = missing_df[missing_df.Missing > 0].sort_values('Missing', ascending=False)
print('Missing values:')
display(missing_df)
print(f'\nDuplicate rows: {train.duplicated().sum()}')

## 3. Target Variable — Credit Score Distribution

In [ ]:
TARGET = 'Credit_Score'
vc = train[TARGET].value_counts()

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
colors = [PALETTE.get(k, '#95a5a6') for k in vc.index]

axes[0].bar(vc.index, vc.values, color=colors, edgecolor='white')
axes[0].set_title('Credit Score Distribution', fontweight='bold')
for i, (k, v) in enumerate(zip(vc.index, vc.values)):
    axes[0].text(i, v + 50, f'{v:,}', ha='center')

axes[1].pie(vc.values, labels=vc.index, colors=colors, autopct='%1.1f%%',
            startangle=140, wedgeprops=dict(edgecolor='white'))
axes[1].set_title('Credit Score Proportions', fontweight='bold')

plt.tight_layout()
plt.savefig(FIG_DIR / '01_target_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

print('\nClass proportions:')
print((vc / len(train) * 100).round(2).to_string())

## 4. Exploratory Data Analysis (EDA)

In [ ]:
# Clean numeric columns
def clean_numeric(s):
    return pd.to_numeric(s.astype(str).str.replace(r'[^0-9.\-]', '', regex=True), errors='coerce')

NUM_COLS = ['Age','Annual_Income','Monthly_Inhand_Salary','Num_Bank_Accounts',
            'Num_Credit_Card','Interest_Rate','Num_of_Loan','Delay_from_due_date',
            'Num_of_Delayed_Payment','Changed_Credit_Limit','Num_Credit_Inquiries',
            'Outstanding_Debt','Credit_Utilization_Ratio','Total_EMI_per_month',
            'Amount_invested_monthly','Monthly_Balance']

for col in NUM_COLS:
    if col in train.columns:
        train[col] = clean_numeric(train[col])
print('Numeric casting done.')

In [ ]:
# Numerical feature distributions
plot_cols = [c for c in NUM_COLS if c in train.columns]
n = len(plot_cols); ncols = 4; nrows = (n + ncols - 1) // ncols

fig, axes = plt.subplots(nrows, ncols, figsize=(16, nrows * 3.5))
axes = axes.flatten()
for i, col in enumerate(plot_cols):
    axes[i].hist(train[col].dropna(), bins=40, color='#3498db', edgecolor='white', alpha=0.85)
    axes[i].set_title(col, fontsize=9, fontweight='bold')
for j in range(i + 1, len(axes)):
    axes[j].set_visible(False)
plt.suptitle('Numerical Feature Distributions', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig(FIG_DIR / '03_numerical_distributions.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Boxplots — key features vs credit score
key_num = ['Annual_Income','Monthly_Inhand_Salary','Outstanding_Debt',
           'Credit_Utilization_Ratio','Num_of_Delayed_Payment',
           'Total_EMI_per_month','Monthly_Balance','Interest_Rate']
key_num = [c for c in key_num if c in train.columns]
order = ['Good', 'Standard', 'Poor']

fig, axes = plt.subplots(2, 4, figsize=(18, 9))
axes = axes.flatten()
for i, col in enumerate(key_num):
    data_list = [train.loc[train[TARGET] == cat, col].dropna() for cat in order]
    bp = axes[i].boxplot(data_list, patch_artist=True,
                         medianprops=dict(color='black', linewidth=2))
    for patch, cat in zip(bp['boxes'], order):
        patch.set_facecolor(PALETTE.get(cat, '#95a5a6'))
        patch.set_alpha(0.8)
    axes[i].set_xticklabels(order)
    axes[i].set_title(col, fontsize=9, fontweight='bold')
plt.suptitle('Key Features vs Credit Score', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(FIG_DIR / '04_boxplots_by_credit_score.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Categorical features vs credit score
cat_cols = ['Occupation', 'Credit_Mix', 'Payment_of_Min_Amount', 'Payment_Behaviour']
cat_cols = [c for c in cat_cols if c in train.columns]

fig, axes = plt.subplots(2, 2, figsize=(16, 10))
axes = axes.flatten()
for i, col in enumerate(cat_cols):
    ct = pd.crosstab(train[col], train[TARGET], normalize='index') * 100
    ct = ct[[c for c in ['Good','Standard','Poor'] if c in ct.columns]]
    colors = [PALETTE.get(c, '#95a5a6') for c in ct.columns]
    ct.plot(kind='bar', ax=axes[i], color=colors, edgecolor='white', width=0.7)
    axes[i].set_title(f'{col} vs Credit Score (%)', fontweight='bold')
    axes[i].tick_params(axis='x', rotation=35)
    axes[i].legend(title='Credit Score', fontsize=8)
plt.suptitle('Categorical Features vs Credit Score', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(FIG_DIR / '05_categorical_vs_credit_score.png', dpi=150, bbox_inches='tight')
plt.show()

## 5. Correlation Heatmap

In [ ]:
label_map = {'Good': 2, 'Standard': 1, 'Poor': 0}
train['Credit_Score_Num'] = train[TARGET].map(label_map)
corr_cols = [c for c in NUM_COLS if c in train.columns] + ['Credit_Score_Num']
corr_mat = train[corr_cols].corr()

fig, ax = plt.subplots(figsize=(14, 11))
mask = np.triu(np.ones_like(corr_mat, dtype=bool))
sns.heatmap(corr_mat, mask=mask, annot=True, fmt='.2f', cmap='RdYlGn',
            center=0, linewidths=0.4, ax=ax, annot_kws={'size': 7})
ax.set_title('Correlation Heatmap', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(FIG_DIR / '06_correlation_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()

# Top correlations with target
target_corr = corr_mat['Credit_Score_Num'].drop('Credit_Score_Num').sort_values()
print('\nCorrelations with Credit Score (numeric):')
print(target_corr.to_string())

## 6. Key Insights — Income, Debt, Payment History

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
insight_cols = ['Annual_Income', 'Outstanding_Debt', 'Num_of_Delayed_Payment']
for ax, col in zip(axes, insight_cols):
    for score, color in PALETTE.items():
        subset = train.loc[train[TARGET] == score, col].dropna()
        subset_clipped = subset.clip(upper=subset.quantile(0.99))
        ax.hist(subset_clipped, bins=40, alpha=0.6, color=color, label=score)
    ax.set_title(col, fontweight='bold')
    ax.legend(title='Credit Score', fontsize=8)
plt.suptitle('Key Financial Features by Credit Score', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

# Summary stats
print('Mean values by Credit Score:')
print(train.groupby(TARGET)[insight_cols].mean().round(2).to_string())

## 7. Feature Engineering & Preprocessing

In [ ]:
# Reload fresh copies
train_raw = pd.read_csv('../data/train.csv', low_memory=False)
test_raw  = pd.read_csv('../data/test.csv',  low_memory=False)

DROP_COLS = ['ID', 'Customer_ID', 'Name', 'SSN', 'Month']
LABEL_MAP = {'Poor': 0, 'Standard': 1, 'Good': 2}
INV_LABEL = {v: k for k, v in LABEL_MAP.items()}
CAT_COLS  = ['Occupation', 'Credit_Mix', 'Payment_of_Min_Amount', 'Payment_Behaviour']

def parse_credit_history_age(series):
    years  = series.astype(str).str.extract(r'(\d+)\s*Year',  expand=False)
    months = series.astype(str).str.extract(r'(\d+)\s*Month', expand=False)
    y = pd.to_numeric(years,  errors='coerce').fillna(0)
    m = pd.to_numeric(months, errors='coerce').fillna(0)
    result = y * 12 + m
    result[result == 0] = np.nan
    return result

def preprocess(df, is_train=True, scaler=None, cat_encoders=None, fit=True):
    df = df.copy()
    df.drop(columns=[c for c in DROP_COLS if c in df.columns], inplace=True)
    for col in NUM_COLS:
        if col in df.columns:
            df[col] = clean_numeric(df[col])
    if 'Credit_History_Age' in df.columns:
        df['Credit_History_Months'] = parse_credit_history_age(df['Credit_History_Age'])
        df.drop(columns=['Credit_History_Age'], inplace=True)
    if 'Type_of_Loan' in df.columns:
        df['Num_Loan_Types'] = df['Type_of_Loan'].fillna('').astype(str).apply(
            lambda x: len([t for t in x.split(',') if t.strip() not in ('', 'nan')])
        )
        df.drop(columns=['Type_of_Loan'], inplace=True)
    eps = 1e-6
    if 'Outstanding_Debt' in df.columns and 'Annual_Income' in df.columns:
        df['Debt_to_Income'] = df['Outstanding_Debt'] / (df['Annual_Income'] + eps)
    if 'Total_EMI_per_month' in df.columns and 'Monthly_Inhand_Salary' in df.columns:
        df['EMI_to_Salary'] = df['Total_EMI_per_month'] / (df['Monthly_Inhand_Salary'] + eps)
    if 'Amount_invested_monthly' in df.columns and 'Monthly_Inhand_Salary' in df.columns:
        df['Savings_Rate'] = df['Amount_invested_monthly'] / (df['Monthly_Inhand_Salary'] + eps)
    if 'Monthly_Balance' in df.columns and 'Monthly_Inhand_Salary' in df.columns:
        df['Balance_Buffer'] = df['Monthly_Balance'] / (df['Monthly_Inhand_Salary'] + eps)
    if 'Num_Credit_Card' in df.columns and 'Num_Bank_Accounts' in df.columns:
        df['Cards_per_Account'] = df['Num_Credit_Card'] / (df['Num_Bank_Accounts'] + eps)
    if cat_encoders is None:
        cat_encoders = {}
    for col in CAT_COLS:
        if col not in df.columns: continue
        df[col] = df[col].astype(str).str.strip().replace({'nan': np.nan, '_': np.nan}).fillna('Unknown')
        if fit:
            le = LabelEncoder()
            df[col] = le.fit_transform(df[col])
            cat_encoders[col] = le
        else:
            le = cat_encoders[col]
            df[col] = df[col].apply(lambda x: x if x in set(le.classes_) else le.classes_[0])
            df[col] = le.transform(df[col])
    obj_cols = df.select_dtypes(include='object').columns.tolist()
    if is_train and TARGET in obj_cols: obj_cols.remove(TARGET)
    df.drop(columns=obj_cols, inplace=True)
    feature_cols = [c for c in df.columns if c != TARGET]
    df[feature_cols] = df[feature_cols].replace([np.inf, -np.inf], np.nan)
    for col in feature_cols:
        if df[col].isnull().any():
            med = df[col].median()
            df[col] = df[col].fillna(0 if pd.isna(med) else med)
    feature_cols = [c for c in df.columns if c != TARGET]
    if fit:
        scaler = StandardScaler()
        df[feature_cols] = scaler.fit_transform(df[feature_cols])
    else:
        df[feature_cols] = scaler.transform(df[feature_cols])
    df[feature_cols] = df[feature_cols].replace([np.inf, -np.inf], 0).fillna(0)
    return df, scaler, cat_encoders

train_proc, scaler, cat_encoders = preprocess(train_raw, is_train=True, fit=True)
test_proc,  _,      _            = preprocess(test_raw,  is_train=False, scaler=scaler,
                                               cat_encoders=cat_encoders, fit=False)
print(f'Train processed: {train_proc.shape}')
print(f'Test  processed: {test_proc.shape}')
print(f'Features: {[c for c in train_proc.columns if c != TARGET]}')

## 8. Model Training & Evaluation

In [ ]:
X = train_proc.drop(columns=[TARGET])
y = train_proc[TARGET].map(LABEL_MAP)

X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print(f'X_train: {X_train.shape}, X_val: {X_val.shape}')

models = {
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42, C=0.5),
    'Decision Tree':       DecisionTreeClassifier(max_depth=8, random_state=42),
    'Random Forest':       RandomForestClassifier(n_estimators=200, max_depth=12,
                                                   random_state=42, n_jobs=-1),
}

results = {}
for name, model in models.items():
    model.fit(X_train, y_train)
    y_pred  = model.predict(X_val)
    y_proba = model.predict_proba(X_val)
    acc  = accuracy_score(y_val, y_pred)
    prec = precision_score(y_val, y_pred, average='weighted', zero_division=0)
    rec  = recall_score(y_val, y_pred, average='weighted', zero_division=0)
    f1   = f1_score(y_val, y_pred, average='weighted', zero_division=0)
    auc  = roc_auc_score(y_val, y_proba, multi_class='ovr', average='weighted')
    cv   = cross_val_score(model, X, y, cv=StratifiedKFold(5), scoring='accuracy').mean()
    results[name] = {'Accuracy': acc, 'Precision': prec, 'Recall': rec,
                     'F1-Score': f1, 'ROC-AUC': auc, 'CV-Accuracy': cv,
                     'model': model, 'y_pred': y_pred, 'y_proba': y_proba}
    print(f'\n{name}:')
    print(classification_report(y_val, y_pred,
          target_names=[INV_LABEL[i] for i in sorted(INV_LABEL)]))

## 9. Model Comparison

In [ ]:
metrics_df = pd.DataFrame({
    name: {k: round(v, 4) for k, v in info.items()
           if k not in ('model','y_pred','y_proba')}
    for name, info in results.items()
}).T

display(metrics_df)

metric_names = ['Accuracy','Precision','Recall','F1-Score','ROC-AUC']
x = np.arange(len(metric_names))
width = 0.25
colors_bar = ['#3498db', '#e67e22', '#2ecc71']

fig, ax = plt.subplots(figsize=(13, 6))
for i, (name, info) in enumerate(results.items()):
    vals = [info[m] for m in metric_names]
    bars = ax.bar(x + i * width, vals, width, label=name,
                  color=colors_bar[i], edgecolor='white', alpha=0.88)
    for bar in bars:
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
                f'{bar.get_height():.3f}', ha='center', va='bottom', fontsize=7.5)
ax.set_xticks(x + width)
ax.set_xticklabels(metric_names)
ax.set_ylim(0, 1.12)
ax.set_title('Model Performance Comparison', fontsize=14, fontweight='bold')
ax.legend()
plt.tight_layout()
plt.savefig(FIG_DIR / '12_model_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

## 10. Confusion Matrices

In [ ]:
class_names = [INV_LABEL[i] for i in sorted(INV_LABEL)]
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for ax, (name, info) in zip(axes, results.items()):
    cm = confusion_matrix(y_val, info['y_pred'])
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=class_names, yticklabels=class_names, ax=ax)
    ax.set_title(f'{name}\nConfusion Matrix', fontweight='bold')
    ax.set_xlabel('Predicted'); ax.set_ylabel('Actual')
plt.tight_layout()
plt.savefig(FIG_DIR / '13_confusion_matrices.png', dpi=150, bbox_inches='tight')
plt.show()

## 11. ROC Curves

In [ ]:
y_val_bin = label_binarize(y_val, classes=[0, 1, 2])
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for ax, (name, info) in zip(axes, results.items()):
    for i, (cls, ls, lc) in enumerate(zip(
            class_names, ['-','--','-.'], ['#e74c3c','#3498db','#2ecc71'])):
        fpr, tpr, _ = roc_curve(y_val_bin[:, i], info['y_proba'][:, i])
        auc_cls = roc_auc_score(y_val_bin[:, i], info['y_proba'][:, i])
        ax.plot(fpr, tpr, linestyle=ls, color=lc,
                label=f'{cls} (AUC={auc_cls:.2f})', linewidth=1.8)
    ax.plot([0,1],[0,1],'k--', linewidth=0.8)
    ax.set_title(f'{name}\nROC Curve', fontweight='bold')
    ax.set_xlabel('FPR'); ax.set_ylabel('TPR')
    ax.legend(fontsize=8)
plt.tight_layout()
plt.savefig(FIG_DIR / '14_roc_curves.png', dpi=150, bbox_inches='tight')
plt.show()

## 12. Feature Importance

In [ ]:
rf_model = results['Random Forest']['model']
fi = pd.Series(rf_model.feature_importances_, index=X.columns).sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(10, 8))
fi.head(20).sort_values().plot(kind='barh', ax=ax, color='#8e44ad', edgecolor='white')
ax.set_title('Top 20 Feature Importances (Random Forest)', fontsize=13, fontweight='bold')
ax.set_xlabel('Importance Score')
plt.tight_layout()
plt.savefig(FIG_DIR / '15_feature_importance.png', dpi=150, bbox_inches='tight')
plt.show()

print('Top 10 features:')
print(fi.head(10).to_string())

## 13. Final Model & Test Predictions

In [ ]:
best_name  = metrics_df['F1-Score'].astype(float).idxmax()
best_model = results[best_name]['model']
print(f'Best model: {best_name}')
print(f'F1-Score  : {metrics_df.loc[best_name, "F1-Score"]}')
print(f'ROC-AUC   : {metrics_df.loc[best_name, "ROC-AUC"]}')

# Retrain on full data
best_model.fit(X, y)

X_test = test_proc[[c for c in X.columns if c in test_proc.columns]]
for col in X.columns:
    if col not in X_test.columns: X_test[col] = 0
X_test = X_test[X.columns].fillna(0)

y_test_pred  = best_model.predict(X_test)
y_test_proba = best_model.predict_proba(X_test)
pred_labels  = [INV_LABEL[p] for p in y_test_pred]

pred_df = test_raw[['ID', 'Customer_ID']].copy()
pred_df['Predicted_Credit_Score'] = pred_labels
pred_df['Prob_Poor']     = y_test_proba[:, 0].round(4)
pred_df['Prob_Standard'] = y_test_proba[:, 1].round(4)
pred_df['Prob_Good']     = y_test_proba[:, 2].round(4)

pred_df.to_csv('../outputs/predictions/test_predictions.csv', index=False)
print(f'\nPredictions saved.')
print('Prediction distribution:')
print(pred_df['Predicted_Credit_Score'].value_counts().to_string())
pred_df.head(10)

In [ ]:
# Predicted distribution plot
pal = {'Good': '#2ecc71', 'Standard': '#f39c12', 'Poor': '#e74c3c'}
vc = pred_df['Predicted_Credit_Score'].value_counts()
fig, ax = plt.subplots(figsize=(8, 5))
colors = [pal.get(k, '#95a5a6') for k in vc.index]
ax.bar(vc.index, vc.values, color=colors, edgecolor='white')
for i, (k, v) in enumerate(zip(vc.index, vc.values)):
    ax.text(i, v + 10, str(v), ha='center', fontsize=10)
ax.set_title('Predicted Credit Score Distribution (Test Set)', fontweight='bold')
plt.tight_layout()
plt.savefig(FIG_DIR / '16_test_predictions_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

## Summary

| Model | Accuracy | Precision | Recall | F1-Score | ROC-AUC |
|---|---|---|---|---|---|
| Logistic Regression | 0.5990 | 0.5959 | 0.5990 | 0.5818 | 0.7370 |
| Decision Tree | 0.7091 | 0.7158 | 0.7091 | 0.7102 | 0.8313 |
| **Random Forest** | **0.7333** | **0.7373** | **0.7333** | **0.7344** | **0.8583** |

### Key Findings
- **Best Model**: Random Forest (F1=0.7344, ROC-AUC=0.8583)
- **Top Predictors**: Outstanding Debt, Credit History, Delayed Payments, EMI burden
- **Class Imbalance**: Standard (53%), Poor (29%), Good (18%) — handled via stratified splits
- **Feature Engineering**: Debt-to-Income, EMI-to-Salary, and Savings Rate significantly boost performance
- **50,000 test predictions** saved to `outputs/predictions/test_predictions.csv`